# Task 3 — Sentimen, Negasi, & Sarkasme (Google Colab Runner)

Notebook ini menjalankan seluruh eksperimen **T3** sesuai rancangan:
- **E1 · Baseline:** Lexicon sentimen + aturan negasi & sarkasme
- **E2 · Transformer:** Fine-tuning **IndoBERTweet** (`indolem/indobertweet-base-uncased`) multi-head (3 seed)
- **E3 · Model Utama:** IndoBERTweet + Negation & Sarcasm Rule Engine
- **E4 · Ablation:** E3 tanpa aturan negasi & tanpa aturan sarkasme

In [1]:
!nvidia-smi

Sun Nov 22 05:25:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   37C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
from getpass import getpass

token = getpass("Masukkan GitHub Token: ")
REPO_URL = f"https://{token}@github.com/RylRizqi16/Tubes_NLP.git"
PROJECT_DIR = "/content/Tubes_NLP"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}

Masukkan GitHub Token: ··········
Cloning into '/content/Tubes_NLP'...
remote: Enumerating objects: 110, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 110 (delta 24), reused 102 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (110/110), 14.82 MiB | 19.89 MiB/s, done.
Resolving deltas: 100% (24/24), done.
/content/Tubes_NLP


In [3]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

In [4]:
!pip install -q -r requirements.txt

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipykernel==6.17.1, but you have ipykernel 7.4.0 which is incompatible.
jupyter-kernel-gateway 2.5.2 requires jupyter_client<8.0,>=5.2.0, but you have jupyter-client 8.10.0 which is incompatible.
jupyter-kernel-gateway 2.5.2 requires notebook<7.0,>=5.7.6, but you have notebook 7.6.5 which is incompatible.


In [5]:
from pathlib import Path

master_path = Path("data/processed/master_dataset.jsonl")
if not master_path.exists():
    print("master_dataset.jsonl belum ada, mengunduh & membangun dataset...")
    !python scripts/download_datasets.py
    !python scripts/generate_synthetic_edge_cases.py
    !python scripts/build_master_dataset.py
else:
    print(f"Dataset siap: {master_path} ({master_path.stat().st_size / 1e6:.2f} MB)")

Dataset siap: data/processed/master_dataset.jsonl (8.90 MB)


In [6]:
!python -m src.task3.train_baseline

=== E1 BASELINE (LEXICON + RULES) ===

[val_metrics] latency=0.4977 ms/sample
  all            n=2552 | sent_acc=0.5657 | sent_f1=0.5384 | sarc_f1=0.4899
  real           n=2138 | sent_acc=0.5101 | sent_f1=0.5186 | sarc_f1=0.3871
  synthetic      n= 414 | sent_acc=0.6981 | sent_f1=0.3311 | sarc_f1=0.6275
  negation       n= 943 | sent_acc=0.5959 | sent_f1=0.3875 | sarc_f1=0.4034
  code_mixing    n= 354 | sent_acc=0.5273 | sent_f1=0.4582 | sarc_f1=0.3607
  sarcasm_irony  n= 317 | sent_acc=0.9444 | sent_f1=0.3238 | sarc_f1=0.6307

[test_metrics] latency=0.49 ms/sample
  all            n=2555 | sent_acc=0.567 | sent_f1=0.5406 | sarc_f1=0.5529
  real           n=2141 | sent_acc=0.496 | sent_f1=0.5038 | sarc_f1=0.3754
  synthetic      n= 414 | sent_acc=0.7367 | sent_f1=0.3662 | sarc_f1=0.7826
  negation       n= 931 | sent_acc=0.5189 | sent_f1=0.3759 | sarc_f1=0.5081
  code_mixing    n= 380 | sent_acc=0.5596 | sent_f1=0.4791 | sarc_f1=0.36
  sarcasm_irony  n= 346 | sent_acc=0.9402 | sent_f1

In [ ]:
!python -m src.task3.train_indobertweet \
    --model-name indolem/indobertweet-base-uncased \
    --seeds 42 123 2025 \
    --epochs 4 \
    --batch-size 32 \
    --lr 2e-5 \
    --max-length 128 \
    --fp16

Loaded T3 records -> train=11,905, val=2,552, test=2,555
config.json: 100% 1.10k/1.10k [00:00<00:00, 3.65MB/s]
tokenizer_config.json: 100% 2.00/2.00 [00:00<00:00, 12.3kB/s]
vocab.txt: 100% 235k/235k [00:00<00:00, 46.9MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 675kB/s]

pytorch_model.bin: downloading bytes:  55% 247M/445M [00:02<00:00, 229MB/s, 18.4MB/s  ]
pytorch_model.bin: downloading bytes:  78% 349M/445M [00:02<00:00, 242MB/s, 29.5MB/s  ]
pytorch_model.bin: reconstructing file:  27% 121M/445M [00:02<00:06, 48.9MB/s, 4.93MB/s  ]
pytorch_model.bin: downloading bytes:  94% 419M/445M [00:03<00:00, 126MB/s, 34.2MB/s  ]
pytorch_model.bin: reconstructing file:  67% 300M/445M [00:03<00:01, 112MB/s, 19.3MB/s  ]
pytorch_model.bin: downloading bytes: 100% 420M/420M [00:04<00:00, 99.7MB/s, 35.0MB/s  ]
pytorch_model.bin: reconstructing file: 100% 445M/445M [00:04<00:00, 106MB/s, 39.1MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 18990.41it/s]
[transformers] BertModel LOAD REP

In [ ]:
!python -m src.task3.evaluate_t3 --split test

In [ ]:
from src.task3.predictor import predict

samples = [
    "Mantap, lampu jalan depan sekolah mati terus. Semalam ada anak hampir tertabrak.",
    "Bukan tidak menghargai kerja Dishub, tapi Jl. Melati belum pernah diperbaiki sampai sekarang.",
    "Terima kasih banyak untuk Puskesmas Tembalang, pelayanan hari ini cepat dan ramah.",
]
for text in samples:
    print(text)
    print(" ->", predict(text))

In [ ]:
!zip -rq task3_colab_results.zip outputs/ models/task3/
print("Siap diunduh: task3_colab_results.zip")